In [0]:
from pyspark.sql import SparkSession

def create_spark_session(app_name: str = "CineClubWrapped") -> SparkSession:
    return (
        SparkSession.builder
        .appName(app_name)
        # .config(...)  # cluster / warehouse configs if needed
        .getOrCreate()
    )


In [0]:
from pyspark.sql import functions as F

today = F.current_date()

df_T1 = (
    spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")
    .filter(
           (F.col("snapshot_date_id") == "20260622") & (F.col("recognitions_left")>0)
    )
    .select("cde_id", "renewal_status", "plan_id", "recognitions_left", "annual_current_term_end_datetime")
)


In [0]:
from pyspark.sql.window import Window

# Get the single most recent opt-in record per member
w_opt = Window.partitionBy("CDE_ID").orderBy(F.col("timestamp").desc())

df_opt_latest = (
    spark.table("cpx_dataanalytics_gold.marketing.mkt_adobe_push_opt_in_details")
    .withColumn("rn", F.row_number().over(w_opt))
    .filter(F.col("rn") == 1)
    .select(
        F.col("CDE_ID").alias("opt_cde_id"),
        F.col("push_opt_in")
    )
)

# Left join onto df_T1 — members without an opt record will have push_opt_in = null
df_T1 = (
    df_T1
    .join(df_opt_latest, df_T1["cde_id"] == F.col("opt_cde_id"), "left")
    .drop("opt_cde_id")
)

display(df_T1)

In [0]:
display(df_T1)

In [0]:
df_T1.filter(F.col("annual_current_term_end_datetime")<"2026-06-24")

In [0]:
from pyspark.sql import functions as F

# Goal: show how many tickets are CREDITED in each calendar month from July 14 2026.
# A ticket is credited on the renewal date (same day-of-month as term end),
# every month BEFORE the expiry. The expiry date itself is NOT a credit.
#
# Approach:
#   1. Per member: find the first credit date on/after July 14, and count remaining credits
#   2. Explode into one row per credit event using add_months(first_credit, offset)
#   3. Aggregate by the calendar month the credit falls in

start_date = F.to_date(F.lit("2026-07-14"))

# Step 1 — per-member: compute renewal_day, ticket count, and first credit date
df_members = (
    df_T1
    .filter(F.col("plan_id").isin("annual1", "gift-plan"))
    .filter(F.col("renewal_status").like("Active%"))
    .filter(F.col("annual_current_term_end_datetime").cast("date") >= start_date)
    .withColumn("end_date", F.col("annual_current_term_end_datetime").cast("date"))
    .withColumn("renewal_day", F.dayofmonth(F.col("end_date")))
    .withColumn(
        "tickets_from_jul14",
        F.floor(
            F.months_between(
                F.date_add(F.col("end_date"), -1),   # exclude expiry date
                F.date_add(start_date, -1)            # July 13 — include July 14 renewals
            )
        ).cast("long")
    )
    .filter(F.col("tickets_from_jul14") > 0)  # drop members with no upcoming credits
    # First credit = renewal_day in July if >= 14, else in August
    .withColumn(
        "july_renewal",
        F.make_date(F.lit(2026), F.lit(7), F.col("renewal_day"))
    )
    .withColumn(
        "first_credit",
        F.when(F.col("july_renewal") >= start_date, F.col("july_renewal"))
         .otherwise(F.add_months(F.col("july_renewal"), 1))
    )
)

# Step 2 — explode: one row per monthly credit event per member
df_credits = (
    df_members
    .withColumn(
        "month_offset",
        F.explode(F.sequence(F.lit(0), (F.col("tickets_from_jul14") - 1).cast("int")))
    )
    .withColumn("credit_date", F.add_months(F.col("first_credit"), F.col("month_offset")))
    .withColumn("credit_month_sort", F.date_format(F.col("credit_date"), "yyyyMM").cast("int"))
    .withColumn("credit_month",      F.date_format(F.col("credit_date"), "MMMM yyyy"))
)

# Step 3 — aggregate: tickets credited per calendar month
df_tickets_by_credit_month = (
    df_credits
    .groupBy("credit_month", "credit_month_sort")
    .agg(F.count("cde_id").alias("Tickets_Credited"))
    .orderBy("credit_month_sort")
    .drop("credit_month_sort")
)

display(df_tickets_by_credit_month)

# DataFrame Setup for Tickets and Concessions Calculation

In [0]:
df_tickets_per_member = (
    df_members
    .select("cde_id", "plan_id", "renewal_status", "end_date", "renewal_day", "first_credit", "tickets_from_jul14")
    .orderBy(F.col("tickets_from_jul14").desc())
)

display(df_tickets_per_member)